In [1]:
import os
from tkinter import Tk, filedialog, Label, Button, OptionMenu, StringVar, Frame, Canvas, Text, Scrollbar, END
from openpyxl import load_workbook
import csv
import requests
from PIL import Image, ImageTk
import io

def load_excel(file_path):
    workbook = load_workbook(file_path)
    sheet = workbook.active
    return sheet

def load_csv(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        data = list(reader)
    return data

def display_image(url, menu_name):
    try:
        response = requests.get(url)
        img_data = response.content
        img = Image.open(io.BytesIO(img_data))
        img = img.resize((400, 400), Image.LANCZOS)
        
        photo = ImageTk.PhotoImage(img)
        canvas.delete("all")
        canvas.create_image(0, 0, anchor="nw", image=photo)
        canvas.image = photo
        
        image_title.configure(text=menu_name)
        log_message("이미지를 성공적으로 불러왔습니다.")
    except:
        canvas.delete("all")
        log_message("이미지를 불러오는 데 실패했습니다.")

def update_subcategories(*args):
    selected_category = category_var.get()
    subcategories = list(dict.fromkeys([row[1] for row in csv_data[1:] if row[0] == selected_category]))
    subcategory_dropdown['menu'].delete(0, 'end')
    for subcategory in subcategories:
        subcategory_dropdown['menu'].add_command(label=subcategory, command=lambda value=subcategory: subcategory_var.set(value))
    subcategory_var.set("중분류 선택")
    log_message(f"선택한 대분류: {selected_category}")

def update_subsubcategories(*args):
    selected_category = category_var.get()
    selected_subcategory = subcategory_var.get()
    subsubcategories = [row[2] for row in csv_data[1:] if row[0] == selected_category and row[1] == selected_subcategory]
    subsubcategory_dropdown['menu'].delete(0, 'end')
    for subsubcategory in subsubcategories:
        subsubcategory_dropdown['menu'].add_command(label=subsubcategory, command=lambda value=subsubcategory: subsubcategory_var.set(value))
    subsubcategory_var.set("소분류 선택")
    log_message(f"선택한 중분류: {selected_subcategory}")

def update_detailed_menus(*args):
    selected_category = category_var.get()
    selected_subcategory = subcategory_var.get()
    selected_subsubcategory = subsubcategory_var.get()
    
    detailed_menus = [row[3] for row in csv_data[1:] if row[0] == selected_category and row[1] == selected_subcategory and row[2] == selected_subsubcategory]
    detailed_menu_dropdown['menu'].delete(0, 'end')
    for menu in detailed_menus:
        menu_items = menu.split(',')
        for item in menu_items:
            detailed_menu_dropdown['menu'].add_command(label=item.strip(), command=lambda value=item.strip(): detailed_menu_var.set(value))
    detailed_menu_var.set("상세메뉴 선택")
    log_message(f"선택한 소분류: {selected_subsubcategory}")

def select_detailed_menu():
    selected_menu = detailed_menu_var.get()
    if selected_menu != "상세메뉴 선택":
        if sheet:
            excel_data = sheet.iter_rows(min_row=2, values_only=True)
            for row in excel_data:
                if selected_menu == row[0]:
                    url = row[2]
                    break
            
            if url:
                display_image(url, selected_menu)
                log_message(f"선택한 상세메뉴: {selected_menu}")
                log_message(f"URL: {url}")
                display_menu_attributes(selected_menu)
            else:
                log_message(f"선택한 상세메뉴 '{selected_menu}'의 URL을 찾을 수 없습니다.")
        else:
            log_message("엑셀 파일이 선택되지 않았습니다.")
    else:
        log_message("상세메뉴가 선택되지 않았습니다.")

def select_file():
    file_path = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx")])
    if file_path:
        global sheet
        try:
            sheet = load_excel(file_path)
            file_name = os.path.basename(file_path)
            file_label.configure(text=f"선택한 파일: {file_name}")
            log_message(f"엑셀 파일을 성공적으로 불러왔습니다: {file_name}")
        except:
            file_label.configure(text="파일 로드에 실패했습니다.")
            log_message("엑셀 파일 로드에 실패했습니다.")
            sheet = None
    else:
        log_message("파일이 선택되지 않았습니다.")

def select_attribute_file():
    file_path = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx")])
    if file_path:
        global attribute_workbook
        try:
            attribute_workbook = load_workbook(file_path)
            file_name = os.path.basename(file_path)
            attribute_file_label.configure(text=f"선택한 속성 파일: {file_name}")
            log_message(f"속성 파일을 성공적으로 불러왔습니다: {file_name}")
            create_sheet_buttons()
        except:
            attribute_file_label.configure(text="속성 파일 로드에 실패했습니다.")
            log_message("속성 파일 로드에 실패했습니다.")
            attribute_workbook = None
    else:
        log_message("속성 파일이 선택되지 않았습니다.")

def create_sheet_buttons():
    if attribute_workbook:
        for widget in sheet_button_frame.winfo_children():
            widget.destroy()
        
        for sheet_name in attribute_workbook.sheetnames:
            sheet_button = Button(sheet_button_frame, text=sheet_name, command=lambda name=sheet_name: display_menu_attributes_by_sheet(name))
            sheet_button.pack(side="left", padx=5)
    else:
        log_message("속성 파일이 선택되지 않았습니다.")

def display_menu_attributes_by_sheet(sheet_name):
    selected_menu = detailed_menu_var.get()
    if selected_menu != "상세메뉴 선택":
        if attribute_workbook:
            sheet = attribute_workbook[sheet_name]
            header_row = sheet[1]
            for cell in header_row:
                if cell.value == selected_menu:
                    col_idx = cell.column
                    break
            else:
                attribute_text.delete('1.0', END)
                attribute_text.insert(END, f"{sheet_name} 시트에서 선택한 메뉴의 속성을 찾을 수 없습니다.")
                return
            
            menu_attributes = []
            for row in sheet.iter_rows(min_row=2, values_only=True):
                if row[0] is not None and row[col_idx-1] is not None and row[col_idx-1] != '-':
                    attribute_name = row[0]
                    attribute_value = row[col_idx-1]
                    menu_attributes.append(f"{attribute_name}: {attribute_value}")
            
            if menu_attributes:
                attribute_text.delete('1.0', END)
                attribute_text.insert(END, f"메뉴: {selected_menu}\n")
                attribute_text.insert(END, "\n".join(menu_attributes))
            else:
                attribute_text.delete('1.0', END)
                attribute_text.insert(END, f"{sheet_name} 시트에서 선택한 메뉴의 속성을 찾을 수 없습니다.")
        else:
            attribute_text.delete('1.0', END)
            attribute_text.insert(END, "속성 파일이 선택되지 않았습니다.")
    else:
        attribute_text.delete('1.0', END)
        attribute_text.insert(END, "상세메뉴를 선택해주세요.")

def display_menu_attributes(menu_name):
    if attribute_workbook:
        for sheet_name in attribute_workbook.sheetnames:
            display_menu_attributes_by_sheet(sheet_name)
    else:
        attribute_text.delete('1.0', END)
        attribute_text.insert(END, "속성 파일이 선택되지 않았습니다.")

def log_message(message):
    log_text.insert(END, message + "\n")
    log_text.see(END)

# CSV 파일 경로
csv_file_path = "식당대12중53소132상세메뉴380분류.csv"

# CSV 파일 로드
csv_data = load_csv(csv_file_path)

# Tkinter 윈도우 생성
root = Tk()
root.title("메뉴 선택")
root.geometry("1200x800")

# 왼쪽 프레임
left_frame = Frame(root)
left_frame.pack(side="left", padx=10, pady=10)

# 파일 선택 버튼
file_button = Button(left_frame, text="파일 선택", command=select_file)
file_button.pack()

# 선택한 파일 경로 표시
file_label = Label(left_frame, text="")
file_label.pack()

# 메뉴 선택 프레임
menu_frame = Frame(left_frame)
menu_frame.pack()

# 대분류 선택
category_var = StringVar(menu_frame)
category_var.set("대분류 선택")
category_dropdown = OptionMenu(menu_frame, category_var, *list(set([row[0] for row in csv_data[1:]])))
category_dropdown.pack(side="left", padx=5)
category_var.trace('w', update_subcategories)

# 중분류 선택
subcategory_var = StringVar(menu_frame)
subcategory_var.set("중분류 선택")
subcategory_dropdown = OptionMenu(menu_frame, subcategory_var, '')
subcategory_dropdown.pack(side="left", padx=5)
subcategory_var.trace('w', update_subsubcategories)

# 소분류 선택
subsubcategory_var = StringVar(menu_frame)
subsubcategory_var.set("소분류 선택")
subsubcategory_dropdown = OptionMenu(menu_frame, subsubcategory_var, '')
subsubcategory_dropdown.pack(side="left", padx=5)
subsubcategory_var.trace('w', update_detailed_menus)

# 상세메뉴 선택
detailed_menu_var = StringVar(menu_frame)
detailed_menu_var.set("상세메뉴 선택")
detailed_menu_dropdown = OptionMenu(menu_frame, detailed_menu_var, '')
detailed_menu_dropdown.pack(side="left", padx=5)

# 상세메뉴 선택 버튼
Button(left_frame, text="상세메뉴 선택", command=select_detailed_menu).pack()

# 로그 프레임
log_frame = Frame(left_frame)
log_frame.pack(padx=10, pady=10, fill="both", expand=True)

# 로그 텍스트 박스
log_text = Text(log_frame)
log_text.pack(side="left", fill="both", expand=True)

# 로그 스크롤바
log_scrollbar = Scrollbar(log_frame)
log_scrollbar.pack(side="right", fill="y")

# 로그 텍스트 박스와 스크롤바 연결
log_text.config(yscrollcommand=log_scrollbar.set)
log_scrollbar.config(command=log_text.yview)

# 속성 파일 선택 버튼
attribute_button = Button(left_frame, text="속성 파일 선택", command=select_attribute_file)
attribute_button.pack()

# 선택한 속성 파일 경로 표시
attribute_file_label = Label(left_frame, text="")
attribute_file_label.pack()

# 시트 선택 버튼 프레임
sheet_button_frame = Frame(left_frame)
sheet_button_frame.pack()

# 오른쪽 프레임
right_frame = Frame(root)
right_frame.pack(side="right", padx=10, pady=10)

# 이미지 타이틀 표시
image_title = Label(right_frame, text="")
image_title.pack()

# 이미지 표시
canvas = Canvas(right_frame, width=400, height=400)
canvas.pack()

# 메뉴 속성 표시
attribute_text = Text(right_frame, height=10, width=40)
attribute_text.pack()

root.mainloop()